In [37]:
"""Self-contained train/test script: GPU training with categorical data, batched DMatrix."""

import json
import math

import numpy as np
import pandas as pd
import xgboost as xgb
from typing import Optional

SEED           = 42
N_SAMPLES      = 2000
N_FEATURES     = 3
N_CAT_FEATURES = 3   # store_type (4 levels), assortment (3 levels), day_of_week (7 levels)
HORIZON        = 7
TRAIN_FRAC     = 0.8
BATCH_SIZE     = 128

PARAMS = {
    "n_estimators":          300,
    "early_stopping_rounds": 30,
    "max_depth":             5,
    "learning_rate":         0.05,
    "subsample":             0.8,
    "colsample_bytree":      0.8,
    "min_child_weight":      5,
    "reg_alpha":             0.1,
    "reg_lambda":            1.0,
    "tree_method":           "hist",
    "device":                "cuda",
    "objective":             "reg:squarederror",
    "eval_metric":           "rmse",
    "verbosity":             0,
}

def build_dataset(
    seed, n_samples, n_features, n_cat_features, horizon
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    rng = np.random.default_rng(seed)
    X_cont = np.cumsum(rng.standard_normal((n_samples, n_features)), axis=0)
    W = rng.standard_normal((n_features, horizon))
    y_arr = X_cont @ W + rng.standard_normal((n_samples, horizon)) * 0.1

    cont_cols = [f"num_{i}" for i in range(n_features)]
    cat_names  = ["store_type", "assortment", "day_of_week"]
    cat_levels = (4, 3, 7)

    X = pd.DataFrame(X_cont.astype(float), columns=cont_cols)
    for name, lvl in zip(cat_names, cat_levels):
        X[name] = pd.Categorical(rng.integers(0, lvl, size=n_samples))

    y = pd.DataFrame(y_arr.astype(float), columns=[f"target_{h}" for h in range(horizon)])

    feature_types = ["q"] * n_features + ["c"] * n_cat_features
    return X, y, feature_types

def make_batches(n: int, batch_size: int):
    """Yield (start, end) index pairs for batches of ``batch_size`` rows."""
    for start in range(0, n, batch_size):
        rindex = range(start, min(start + batch_size, n))
        yield list(rindex)


def evaluate(booster: xgb.Booster, dm: xgb.DMatrix) -> dict[str, float]:
    """Return RMSE and RMSPE for the given split."""

    y_pred    = np.asarray(booster.predict(dm), dtype=float)
    y_true    = np.asarray(dm.get_label(), dtype=float)
    residuals = (y_true - y_pred).ravel()
    y_flat    = y_true.ravel()

    rmse = float(np.sqrt(np.mean(residuals ** 2)))

    nonzero = y_flat != 0
    pct_err = residuals[nonzero] / y_flat[nonzero]
    rmspe   = float(np.sqrt(np.mean(pct_err ** 2)) * 100)

    return {"RMSE": rmse, "RMSPE (%)": rmspe}


In [19]:
# --- GPU verification ---
build = xgb.build_info()
cuda_available = build.get("USE_CUDA", False)
print(f"XGBoost build info: USE_CUDA={cuda_available}")
if not cuda_available:
    print("WARNING: XGBoost was not built with CUDA — training will run on CPU.")

print("Building dataset …")
X, y, feature_types = build_dataset(
    SEED, N_SAMPLES, N_FEATURES, N_CAT_FEATURES, HORIZON
)

print(f"Dataset shapes: X={X.shape}, y={y.shape}")
print(X.dtypes)

split   = int(len(X) * TRAIN_FRAC)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Horizon: {HORIZON}")

print(f"\nTraining (batch_size={BATCH_SIZE}, native API) …")


dtrain = xgb.DMatrix(X_train, y_train, feature_types=feature_types, enable_categorical=True)
deval  = xgb.DMatrix(X_val,   y_val,   feature_types=feature_types, enable_categorical=True)
booster = None  # Initial booster for warm start; will be updated after each batch


XGBoost build info: USE_CUDA=True
Building dataset …
Dataset shapes: X=(2000, 6), y=(2000, 7)
num_0           float64
num_1           float64
num_2           float64
store_type     category
assortment     category
day_of_week    category
dtype: object
Train: (1600, 6)  Val: (400, 6)  Horizon: 7

Training (batch_size=128, native API) …


In [34]:
class XGBForecaster():
    
    def __init__(self, params: Optional[dict] = PARAMS):
        
        params = PARAMS.copy()  # Avoid mutating the original PARAMS dict
        self.n_estimators          = int(params.pop("n_estimators"))
        self.early_stopping_rounds = int(params.pop("early_stopping_rounds"))
        self.n_batches       = math.ceil(len(X_train) / BATCH_SIZE)
        self.trees_per_batch = max(1, self.n_estimators // self.n_batches)
        self.train_params = params

        self.booster: xgb.Booster = None  # Initial booster for warm start; will be updated after each batch
        return
    
    def _batch_kwargs(self,
                      batch_index: int,
                      dtrain: xgb.DMatrix,
                      deval: xgb.DMatrix) -> dict:
        """Return kwargs for xgb.train for the current batch."""

        batch_kwargs = dict(
            dtrain=dtrain,
            params=self.train_params,
            num_boost_round=self.trees_per_batch,
            evals=[(deval, "val")] if deval is not None else None,
            verbose_eval=False,
            xgb_model=self.booster,
        )

        # Early stopping only on the last batch
        if deval is not None and batch_index == self.n_batches:
            batch_kwargs["early_stopping_rounds"] = self.early_stopping_rounds

        return batch_kwargs

    def fit(self, dtrain: xgb.DMatrix, deval: Optional[xgb.DMatrix] = None):

        iterator = make_batches(dtrain.num_row(), BATCH_SIZE)
        for i, rindex in enumerate(iterator, start=1):

            dtrain_batch  = dtrain.slice(rindex)
            train_kwargs = self._batch_kwargs(i, dtrain_batch, deval)
            self.booster = xgb.train(**train_kwargs)

            # Log batch progress
            if self.train_params.get("verbosity", 0) > 0:
                print(f"Batch {i:2d}/{self.n_batches:3d}" 
                      f" - rows: {rindex[0]}:{rindex[-1]+1} ({len(rindex)})"
                      f" - cumulative trees: {self.booster.num_boosted_rounds()}")

        return self 
    
    def predict(self, dtest: xgb.DMatrix) -> np.ndarray:
        if self.booster is None:
            raise ValueError("Model has not been trained yet. Call fit() before predict().")
        
        preds = self.booster.predict(dtest)
        preds = np.asarray(preds, dtype=float)
        return preds

In [35]:
model = XGBForecaster()
model.fit(dtrain)

In [39]:
model.predict(deval).shape

(400, 7)

In [36]:

# Final evaluation
train_metrics = evaluate(model, dtrain)
val_metrics   = evaluate(model, deval)

ValueError: operands could not be broadcast together with shapes (11200,) (1600,7) 

In [ ]:

iterator = make_batches(dtrain.num_row(), BATCH_SIZE)
for i, rindex in enumerate(iterator, start=1):

    is_last = i == n_batches
    dtrain_batch  = dtrain.slice(rindex)

    # Build xgb.train kwargs; only apply early stopping on the last batch
    train_kwargs = dict(
        params=params,
        dtrain=dtrain_batch,
        num_boost_round=trees_per_batch,
        evals=[(deval, "val")],
        verbose_eval=False,
        xgb_model=booster,
    )
    if is_last:
        train_kwargs["early_stopping_rounds"] = early_stopping_rounds

    booster = xgb.train(**train_kwargs)

    # Log batch progress
    cfg    = json.loads(booster.save_config())
    device = cfg.get("learner", {}).get("generic_param", {}).get("device", "unknown")
    print(f"Batch {i:2d}/{n_batches:3d} - rows: {rindex[0]}:{rindex[-1]+1} ({len(rindex)})" 
          f" - cumulative trees: {booster.num_boosted_rounds()} - device: {device}")

# Final evaluation
train_metrics = evaluate(booster, X_train, y_train)
val_metrics   = evaluate(booster, X_val,   y_val)

print("\nTrain:", {k: f"{v:.4f}" for k, v in train_metrics.items()})
print("Val  :", {k: f"{v:.4f}" for k, v in val_metrics.items()})
